# Sličnost nogometnih igrača — top-5 europskih liga

Notebook sadrži **samo model sličnosti**. Usporedna analiza metrika (Jaccard/Spearman,
proxy ground-truth, testovi stabilnosti, t-SNE) koja je služila radu je uklonjena.

Sva logika živi u `similarity.py` — isti modul koristi i Streamlit aplikacija, pa
notebook i aplikacija ne mogu razići se u rezultatima.

**Podaci:** `podaci/top5_stats_combined.csv` (izlaz iz `build_dataset.py`).

In [ ]:
import pandas as pd

import similarity as s

prostor = s.ucitaj_prostor()

print(f"igrača: {len(prostor.df)}")
print(f"značajki: {len(prostor.znacajke)}")
print(f"uloga: {len(prostor.nazivi_uloga)}")
print(f"stupac s ligom: {'da' if prostor.ima_lige else 'ne — pokreni build_dataset.py'}")

## Prostor značajki

Sve su vrijednosti **per-90**, dakle mjere stil igre, a ne odigranu minutažu.
Izbačene su značajke koje su točna funkcija drugih zadržanih — inače bi isti aspekt
igre ušao u račun dvaput i time dobio dvostruku težinu.

In [ ]:
print(f"ZADRŽANO ({len(prostor.znacajke)}):")
for f in prostor.znacajke:
    print(f"  {s.NAZIVI.get(f, f):<32} {f}")

print()
print(f"IZBAČENO ({len(s.IZBACENE)}):")
for f, razlog in s.IZBACENE.items():
    print(f"  {f:<38} {razlog}")

## Uloge umjesto pozicija

Skup podataka nema stupac s pozicijom, pa ulogu izvodimo **iz same igre** —
K-Means klasteriranjem standardiziranih značajki. Kako značajke ravnomjerno pokrivaju
napad, vođenje lopte, obranu i dodavanje, klasteri ispadnu prepoznatljivo pozicijski.

Broj klastera bira se automatski, po najboljem silhouette skoru.

In [ ]:
pregled = (prostor.df.groupby('ULOGA')
           .agg(igraca=('NAME', 'size'), prosj_minuta=('MINS', 'mean'))
           .sort_values('igraca', ascending=False))
pregled['prosj_minuta'] = pregled['prosj_minuta'].round(0)
display(pregled)

for uloga in pregled.index:
    primjeri = prostor.df[prostor.df['ULOGA'] == uloga].nlargest(4, 'MINS')['NAME'].tolist()
    print(uloga)
    print('   ', ', '.join(primjeri))

## Objedinjeni model

Četiri izvorne metrike nisu četiri neovisna modela, nego **dva parametra** iste formule:

$$\mathrm{sim}(a,b) = \frac{a^\top M b}{(a^\top M a)^{\alpha/2}\,(b^\top M b)^{\alpha/2}}$$

| | $M = I$ | $M = \mathrm{korr}$ |
|---|---|---|
| $\alpha = 1$ | kosinusna | soft-kosinusna |
| $\alpha = 0$ | skalarni produkt | — |

Euklidska udaljenost nije poseban slučaj, ali na standardiziranim podacima nakon
L2-normalizacije vrijedi $d^2 = 2 - 2\cos$, pa daje **isti poredak** kao kosinusna —
mjeri istu os, samo kao udaljenost umjesto kao kut.

U praksi to znači da $\alpha$ bira između *„sličan stil”* i *„slična količina doprinosa”*,
a $M$ određuje smiju li povezane statistike doprinositi zajedno.

In [ ]:
CILJ = 'Rodri'

usporedba = {}
for kljuc, spec in s.PRESETI.items():
    rezultat = s.slicni_igraci(prostor, CILJ, preset=kljuc, n=5)
    usporedba[spec['naziv']] = rezultat['NAME'].tolist()

print('Najsličniji igrači —', CILJ)
display(pd.DataFrame(usporedba, index=[f'{i}.' for i in range(1, 6)]))

### Kontinuum stil ↔ količina

Presetovi su samo dvije krajnje točke. Parametar $\alpha$ je zapravo klizač, pa se
vidi kako lista prelazi iz *„igra na isti način”* u *„pridonosi u istoj mjeri”*.

In [ ]:
redovi = {}
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    rezultat = s.slicni_igraci(prostor, CILJ, preset=None, alpha=alpha, koristi_M=True, n=5)
    redovi[f'α={alpha}'] = rezultat['NAME'].tolist()

print('α=0 količina  →  α=1 stil')
display(pd.DataFrame(redovi, index=[f'{i}.' for i in range(1, 6)]))

## Traženje sličnih igrača

Glavna funkcija, sa svim filtrima.

In [ ]:
s.slicni_igraci(
    prostor,
    'Bellingham',
    preset='soft_cosine',
    n=10,
    ista_uloga=True,        # ograniči na istu izvedenu ulogu
    izbaci_istu_ligu=False, # zahtijeva stupac LEAGUE
    min_minuta=900,
)

## Zašto su dva igrača slična

Razlaganje po značajkama pokazuje gdje se profili poklapaju, a gdje razilaze
(`z` = standardizirana vrijednost, 0 = prosjek lige).

In [ ]:
par = s.slicni_igraci(prostor, 'Bellingham', n=1)
a = s.pronadi_igraca(prostor, 'Bellingham')[0]
b = s.pronadi_igraca(prostor, par['NAME'].iloc[0])[0]

print(prostor.df.iloc[a]['NAME'], ' ↔ ', prostor.df.iloc[b]['NAME'])
razlika = s.usporedi_profile(prostor, a, b)

print('NAJVEĆE RAZLIKE')
display(razlika.head(5))
print('NAJBLIŽE PODUDARANJE')
display(razlika.tail(5))